In [ ]:
# Copyright 2023 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# GroceryBot - 장보기 & 레시피 도우미 예제 - RAG + ReAct

<!---table align="left">
  <td style="text-align: center">
    <a href="https://colab.research.google.com/github/GoogleCloudPlatform/generative-ai/blob/main/language/use-cases/chatbots/grocerybot_assistant.ipynb">
      <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Google Colaboratory logo"><br> Run in Colab
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/GoogleCloudPlatform/generative-ai/blob/main/language/use-cases/chatbots/grocerybot_assistant.ipynb">
      <img src="https://img.shields.io/badge/GitHub-View_on_GitHub-blue?logo=github" alt="GitHub logo"><br> View on GitHub
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/agent-platform/workbench/deploy-notebook?download_url=https://raw.githubusercontent.com/GoogleCloudPlatform/generative-ai/main/language/use-cases/chatbots/grocerybot_assistant.ipynb">
      <img src="https://upload.wikimedia.org/wikipedia/commons/1/1d/Google_Gemini_icon_2025.svg" height=32 width=32 alt="Agent Platform logo"><br> Open in Agent Platform Workbench
    </a>
  </td>
</table--->


| | |
|-|-|
|Author(s) | [Elia Secchi](https://github.com/eliasecchig) |
|Upgrader | [A.Mahdy](https://github.com/amahdy) |
|Upgrader | [P.Leroy](https://github.com/paulleroyza) |


## 개요
이 노트북은 검색 증강 생성(RAG, Retrieval Augmented Generation)과 추론+행동(ReAct, Reasoning + Acting)을 활용해, 고객의 장보기 과정을 도와주는 대화형 봇을 만드는 방법을 보여 줍니다.

두 접근법에 대해 더 알고 싶다면 관련 논문을 참고하세요: [RAG arXiv 논문](https://arxiv.org/pdf/2005.11401.pdf) & [ReAct arXiv 논문](https://arxiv.org/abs/2210.03629.pdf)

## 시나리오
여러분이 즐겨 찾는 식료품점 Cymbal Grocery의 고객이라고 상상해 봅시다. 저녁으로 라자냐 같은 근사한 요리를 만들고 싶지만, 어디서부터 시작해야 할지, 어떤 재료를 사야 할지, 라자냐를 어떻게 만드는지 모릅니다.

웹사이트에 들어가 보니 Cymbal Grocery가 새로운 대화형 봇 GroceryBot을 막 출시했네요!

GroceryBot은 다음과 같은 방식으로 장보기 과정을 도와줍니다.

1. 레시피를 추천해 줍니다
2. 재료 목록과 조리법을 알려 줍니다
3. 그 레시피에 필요한, 구매하면 좋을 상품을 추천해 줍니다
4. 저녁 식사를 위해 새로 사고 싶은 상품을 찾도록 도와줍니다!

## 목표 및 요구 사항
여러분의 목표는 **GroceryBot**을 만드는 것입니다!

핵심 요구 사항이 하나 있습니다. 이 봇이 반드시 **그라운딩(grounded)** 되어야 한다는 점입니다. 그라운딩이란 LLM을 데이터베이스 같은 외부 지식 소스에 연결하는 과정을 말합니다.

실무적으로 이는 GroceryBot이 다음을 활용해야 한다는 뜻입니다.

1. Cymbal Grocery의 기존 레시피 카탈로그. GroceryBot은 이 카탈로그에 없는 레시피를 추천해서는 안 됩니다.
2. Cymbal Grocery의 기존 상품 카탈로그. GroceryBot은 이 카탈로그에 없는 상품을 추천해서는 안 됩니다.
3. 레시피별로 미리 계산해 둔 추천 상품 목록.

이를 위해 RAG(검색 증강 생성) 방식을 사용할 수 있습니다. RAG는 LLM에 보내는 프롬프트에 사실 정보(여기서는 레시피와 상품 정보)를 삽입해 환각(hallucination) 문제를 완화하는 접근법입니다.


아래 이미지는 이 솔루션을 실제로 배포하고 프런트엔드 애플리케이션과 통합했을 때 GroceryBot으로 무엇이 가능한지를 보여 줍니다.

![image](https://storage.googleapis.com/github-repo/img/language/reference_architectures/spotbot/spotbot_chat_example.png)


### GroceryBot 구현


이 시스템은 Agent Platform의 생성형 모델과 LangChain으로 동작합니다. LangChain이 처음이라면 [이 노트북](https://github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/orchestration/intro_langchain_gemini.ipynb)으로 프레임워크에 익숙해지는 것을 권장합니다.

앞서 언급했듯이 모델을 그라운딩하려면 LLM을 사내 데이터베이스에 연결해야 합니다. 이를 위해 LangChain에서 [ReAct 방식](https://ai.googleblog.com/2022/11/react-synergizing-reasoning-and-acting.html)의 에이전트를 구현합니다. 이 에이전트는 스스로 판단해 언제 이 데이터베이스들을 조회할지 결정할 수 있습니다. LangChain의 에이전트에 대해 더 알고 싶다면 [이 페이지](https://python.langchain.com/docs/how_to/#agents)를 참고하세요.

데모 목적이므로 이 노트북에서는 로컬 데이터베이스만 사용합니다. 구성은 다음과 같습니다.
- 상품 및 레시피 카탈로그는 [Faiss](https://python.langchain.com/docs/integrations/vectorstores/faiss/)를 사용해 로컬에 정의합니다. 몇 개의 예시를 넘어 확장이 필요한 운영 환경이라면, ScaNN 기반의 관리형 벡터 데이터베이스인 [Agent Platform Matching Engine](https://cloud.google.com/vertex-ai/docs/matching-engine/overview)을 검토해 보세요. ([ScaNN 유사도 검색](https://ai.googleblog.com/2020/07/announcing-scann-efficient-vector.html))
- 레시피의 상세 정보와 그 레시피에 추천되는 상품 목록도 로컬에 저장합니다. 운영 환경이라면 [Cloud Datastore](https://cloud.google.com/datastore) 같은 NoSQL 데이터베이스에 저장하는 편이 좋습니다.

아래 다이어그램에서 에이전트의 각 구성 요소와 데이터베이스와의 상호작용 흐름을 확인할 수 있습니다.

![image](https://storage.googleapis.com/github-repo/img/language/reference_architectures/spotbot/spotbot_architecture.png)

### 비용

이 튜토리얼은 Google Cloud의 유료 구성 요소를 사용합니다.
- Agent Platform Studio

[Agent Platform 가격 정책](https://cloud.google.com/products/gemini-enterprise-agent-platform/pricing)을 확인하고, [가격 계산기](https://cloud.google.com/products/calculator/)로 예상 사용량에 따른 비용을 산출해 보세요.

# 시작하기

### 라이브러리 설치

In [ ]:
%pip install --upgrade langchain langchain-google-genai google-genai google-generativeai protobuf langchain-community faiss-cpu --user

***Colab 전용***: 커널을 재시작하려면 아래 셀의 주석을 해제해 실행하거나 재시작 버튼을 사용하세요. Agent Platform Workbench에서는 상단의 버튼으로 터미널을 재시작할 수 있습니다.

In [ ]:
# 설치한 패키지를 환경에서 사용할 수 있도록 커널을 자동으로 재시작합니다.
import IPython

app = IPython.Application.instance()
app.kernel.do_shutdown(True)

### 노트북 환경 인증
* **Colab**에서 이 노트북을 실행 중이라면 아래 셀의 주석을 해제하고 진행하세요.
* **Agent Platform Workbench**를 사용 중이라면 [여기](https://github.com/GoogleCloudPlatform/generative-ai/tree/main/setup-env)의 설정 안내를 확인하세요.

In [ ]:
# from google.colab import auth
# auth.authenticate_user()

### 라이브러리 임포트

In [ ]:
from collections.abc import Iterator
import glob
import pprint
from typing import Any
from IPython.display import display, Markdown, clear_output

from langchain.agents import create_agent
from langchain_community.document_loaders import TextLoader
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.documents.base import Document
from langchain.tools import tool, ToolRuntime
from langchain_community.vectorstores import FAISS
from langchain_core.vectorstores.base import VectorStoreRetriever

from langgraph.checkpoint.memory import InMemorySaver
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables import RunnableConfig
from typing import Dict, Any

from tqdm import tqdm

### 모델 초기화

In [ ]:
# 대화형 ReAct 에이전트에 사용할 LLM을 생성합니다.
llm = ChatGoogleGenerativeAI(
    vertexai=True,
    model="gemini-3.1-flash-lite",
    max_output_tokens=256,
    temperature=0,
    top_p=0.8,
    top_k=40,
)

# 텍스트 임베딩 모델을 생성합니다.
embedding = GoogleGenerativeAIEmbeddings(vertexai=True, model="models/gemini-embedding-001")

# 레시피 & 상품 리트리버 만들기

앞서 말했듯이 목표는 폐쇄형 도메인 데이터베이스의 정보를 활용해 LLM에 더 많은 맥락을 제공하는 것입니다. 이를 위해 LangChain에서 두 개의 로컬 벡터 데이터베이스(상품용, 레시피용)와 상호작용하는 리트리버 두 개를 만듭니다.

**최초 1회 작업으로** 모든 상품과 레시피 항목을 임베딩으로 변환해 해당 벡터 데이터베이스에 적재합니다.

**검색 시점에는** 질의(예: 라자냐)를 임베딩으로 변환한 뒤 벡터 유사도 검색을 수행해 질의와 가장 가까운 항목(예: 라자냐 알 포르노, 채식 라자냐)을 찾습니다.

이 두 데이터베이스에는 [Agent Platform 생성형 AI 모델](https://cloud.google.com/vertex-ai/docs/generative-ai/learn/models#foundation_models)로 만들어 낸 더미 레시피와 상품 데이터를 사용합니다.

이 데이터를 VectorStore에 적재할 때는 [LangChain TextLoader](https://api.python.langchain.com/en/latest/document_loaders/langchain_community.document_loaders.text.TextLoader.html)를 사용합니다.

온라인에 공개된 실제 레시피·상품 데이터로 이 방식을 시험해 보고 싶다면 [LangChain WebBaseLoader](https://python.langchain.com/docs/integrations/document_loaders/web_base/)를 사용할 수 있습니다.

**참고:** 카탈로그 데이터(레시피/상품 텍스트 파일)는 영어로 되어 있습니다. 그래서 이 노트북의 한글 질문에는 `lasagne`, `tomato` 같은 영어 키워드를 함께 넣어 검색 정확도를 유지합니다.

먼저 공개 Cloud Storage 버킷에서 상품과 레시피 더미 데이터를 내려받아 로컬에 저장합니다.

In [ ]:
!gcloud storage cp -r "gs://github-repo/use-cases/grocery_bot/*" .

그다음, 상품용과 레시피용 벡터 데이터베이스를 만들기 위한 함수들을 정의합니다.

In [ ]:
def chunks(lst: list[Any], n: int) -> Iterator[list[Any]]:
    """리스트 lst를 크기 n의 청크로 잘라 순서대로 반환합니다.

    Args:
        lst: 청크로 나눌 리스트.
        n: 각 청크의 크기.

    Yields:
        lst에서 가져온 다음 n개 원소로 이루어진 리스트.
    """

    for i in range(0, len(lst), n):
        yield lst[i : i + n]


def load_docs_from_directory(dir_path: str) -> list[Document]:
    """디렉터리에서 여러 문서를 읽어 옵니다.

    Args:
      dir_path: 문서가 들어 있는 디렉터리 경로.

    Returns:
      해당 디렉터리에 있는 문서들의 리스트.
    """

    docs = []
    for file_path in glob.glob(dir_path):
        loader = TextLoader(file_path)
        docs = docs + loader.load()
    return docs


def create_retriever(top_k_results: int, dir_path: str) -> VectorStoreRetriever:
    """문서 디렉터리로부터 리트리버를 생성합니다.

    Args:
        top_k_results: 검색 시 반환할 결과 개수.
        dir_path: 문서 디렉터리 경로.

    Returns:
        생성된 리트리버.
    """

    BATCH_SIZE_EMBEDDINGS = 5
    docs = load_docs_from_directory(dir_path=dir_path)
    doc_chunk = chunks(docs, BATCH_SIZE_EMBEDDINGS)
    for index, chunk in tqdm(enumerate(doc_chunk)):
        if index == 0:
            db = FAISS.from_documents(chunk, embedding)
        else:
            db.add_documents(chunk)

    retriever = db.as_retriever(search_kwargs={"k": top_k_results})
    return retriever

이제 앞 단계에서 정의한 함수로 벡터 DB를 만들 준비가 되었습니다.
각 벡터 DB는 리트리버 인스턴스를 제공합니다. 리트리버는 질의를 받으면 그에 맞는 문서 목록을 돌려주는 파이썬 객체입니다.

다음 두 개를 만듭니다.
- `recipe_retriever`: 질의에 맞는 레시피를 검색
- `product_retriever`: 질의에 맞는 상품을 검색

In [ ]:
recipe_retriever = create_retriever(top_k_results=2, dir_path="./recipes/*")
product_retriever = create_retriever(top_k_results=5, dir_path="./products/*")

이제 리트리버를 테스트해 봅시다! 예를 들어 `recipe_retriever`에 "라자냐 레시피"를 물어보면 질의와 가장 가까운 레시피 두 건이 나와야 합니다.

In [ ]:
docs = recipe_retriever.invoke("라자냐(lasagne) 레시피 있나요?")
pprint.pprint([doc.metadata for doc in docs])

`product_retriever`에 토마토를 물어봐도 비슷하게 동작합니다.

In [ ]:
docs = product_retriever.invoke("토마토(tomatoes) 있나요?")
pprint.pprint([doc.metadata for doc in docs])

`recipe_retriever`는 문서를 두 건만 반환하는 반면 `product_retriever`는 다섯 건을 반환한다는 점에 주목하세요. 각 리트리버가 반환하는 문서 수는 `create_retriever` 함수의 `top_k_results` 파라미터로 조절할 수 있습니다.

## 에이전트

리트리버를 만들었으니, 이제 ReAct 방식을 구현할 LangChain 에이전트를 만들 차례입니다.

에이전트는 여러 도구(tool)를 사용할 수 있습니다. 도구는 여러분이 원하는 어떤 일이든 하도록 만들 수 있는 파이썬 함수라고 생각하면 됩니다. 에이전트 구성의 특별한 점은 사용자 입력에 따라 **스스로** 어떤 도구를 어떤 순서로 호출할지 결정한다는 것입니다.

## 1. 에이전트 도구

가장 먼저 만들어야 할 것은 에이전트가 사용할 도구입니다. 각 도구마다 그 도구가 무엇을 하는지 잘 설명해 주는 것이 매우 중요합니다. 에이전트가 이 설명을 보고 행동을 결정하기 때문입니다.


도구를 만드는 방법은 여러 가지가 있으니 자세한 내용은 [이 문서](https://python.langchain.com/docs/how_to/custom_tools/)를 참고하세요. 이 노트북에서는 `tool` 데코레이터 방식을 사용합니다.

일부 도구에는 데코레이터에 `return_direct=True` 파라미터가 설정되어 있는 것을 볼 수 있습니다. 이렇게 하면 도구의 출력이 LLM에 의해 후처리되지 않고 사용자에게 그대로 전달됩니다.


먼저 앞에서 정의한 두 리트리버 객체(`recipe_retriever`, `product_retriever`)를 활용하는 도구 두 개를 만듭니다.

In [ ]:
@tool(return_direct=True)
def retrieve_recipes(query: str) -> str:
    """
    레시피 카탈로그에서 질의에 맞는 레시피를 검색합니다.
    결과를 추가로 가공하지 말고 그대로 반환하세요.
    """

    #docs = recipe_retriever.get_relevant_documents(query)
    docs = recipe_retriever.invoke(query)

    return (
        f"{query} 관련해서 자세히 살펴볼 레시피를 골라 주세요: "
        + str([doc.metadata for doc in docs])
    )

In [ ]:
@tool(return_direct=True)
def retrieve_products(query: str) -> str:
    """상품 카탈로그에서 질의에 맞는 상품을 검색합니다.
    사용자가 특정 품목으로 구매 가능한 상품을 물어볼 때 사용하세요. 예: `어떤 양파를 살 수 있는지 보여 줄래?`
    """
    docs = product_retriever.invoke(query)
    return (
        f"{query} 관련해서 이런 상품을 찾았어요:  [START CALLBACK FRONTEND] "
        + str([doc.metadata for doc in docs])
        + " [END CALLBACK FRONTEND]"
    )

다음으로 `recipe_selector`를 정의합니다. 사용자가 레시피를 선택하는 행동을 에이전트가 포착하는 데 사용하는 도구입니다. 레시피의 경로가 그 레시피의 식별자로 사용됩니다.

In [ ]:
@tool
def recipe_selector(path: str) -> str:
    """
    사용자가 레시피를 선택했을 때 사용하세요.
    레시피가 선택된 뒤에는 어떤 선택지가 있는지 사용자에게 안내해야 합니다.
    레시피의 재료를 설명하거나, 조리법을 보여 주거나, 카탈로그에서 어떤 상품을 사면 좋을지 추천할 수 있습니다!
    """
    return "좋은 선택이에요! 레시피의 재료를 설명해 드리거나, 조리법을 보여 드리거나, 카탈로그에서 구매하면 좋을 상품을 추천해 드릴 수 있어요!"

네 번째 도구는 레시피 경로를 받아 그 레시피의 상세 정보를 찾아 줍니다. 관찰(observation) 결과로 해당 레시피의 재료와 조리법을 반환하고, 에이전트는 이 정보를 바탕으로 사용자의 구체적인 질문에 답합니다.

In [ ]:
docs = load_docs_from_directory("./recipes/*")
recipes_detail = {doc.metadata["source"]: doc.page_content for doc in docs}


@tool
def get_recipe_detail(path: str) -> str:
    """
    특정 레시피의 재료나 조리 단계 같은 더 자세한 정보를 찾을 때 사용하세요.
    레시피의 재료가 무엇인지, 조리 단계가 어떻게 되는지 알아낼 때 사용합니다.

    출력 예시:
    재료:

    * 라자냐 면 1파운드
    * 소고기 다짐육 1파운드
    * 다진 양파 1/2컵
    * 다진 마늘 2쪽
    * 으깬 토마토 28온스 캔 2개
    * 토마토 소스 15온스 캔 1개
    * 말린 오레가노 1작은술

    카탈로그에서 추천 상품도 보여 드릴까요?
    """
    try:
        return recipes_detail[path]
    except KeyError:
        return "이 레시피의 상세 정보를 찾을 수 없습니다"

마지막으로 특정 레시피에 가장 잘 맞는 상품을 찾아 주는 도구를 정의합니다. 데모 목적이므로 이 정보는 딕셔너리에 하드코딩되어 있습니다.


In [ ]:
@tool(return_direct=True)
def get_suggested_products_for_recipe(recipe_path: str) -> str:
    """사용자가 특정 레시피와 관련된 상품을 구매하려고 할 때만 사용하세요. 예: '라자냐에 필요한 상품을 알려 줄래?'

    Args:
        recipe_path: 레시피 경로.

    Returns:
        사용자가 구매하고 싶어 할 만한 상품 목록.
    """
    recipe_to_product_mapping = {
        "./recipes/lasagne.txt": [
            "./products/angus_beef_lean_mince.txt",
            "./products/large_onions.txt",
            "./products/classic_carrots.txt",
            "./products/classic_tomatoes.txt",
        ]
    }

    try:
      return (
        "이 레시피에 추천하는 재료입니다 [START CALLBACK FRONTEND] "
        + str(recipe_to_product_mapping[recipe_path])
        + " [END CALLBACK FRONTEND]"
      )
    except KeyError:
        return "이 레시피에 해당하는 상품을 찾을 수 없습니다"

## 에이전트 만들기

도구를 모두 정의했으니 이제 에이전트를 만들 차례입니다. 대화가 이어지도록 에이전트에 메모리를 제공합니다.

에이전트는 범용 에이전트로 초기화됩니다. 더 알고 싶다면 [관련 문서](https://python.langchain.com/docs/modules/agents/agent_types/chat_conversation_agent)와 [다른 에이전트 유형](https://python.langchain.com/docs/modules/agents/agent_types/)을 참고하세요.

In [ ]:
checkpointer = InMemorySaver()
tools = [
    retrieve_recipes,
    retrieve_products,
    get_recipe_detail,
    get_suggested_products_for_recipe,
    recipe_selector,
]

agent = create_agent(
    llm,
    tools,
    checkpointer = checkpointer
)

# 프롬프트 전달과 출력을 한 줄로 처리하는 간단한 래퍼를 정의합니다.
def wrapped_agent(query,agent=agent):
    """에이전트 호출을 간결하게 만들어 주는 래퍼"""
    response = agent.invoke({"messages": [{"role": "user", "content": query}]},{"configurable":{"thread_id":"1"}})
    content = response.get("messages")[-1].content
    if isinstance(content, str):
        return Markdown(content)
    elif isinstance(content, list):
        return Markdown(content[0].get("text"))
    else:
        return(content)

### 라자냐를 만들어 봅시다!

In [ ]:
wrapped_agent("라자냐(lasagne)를 만들고 싶어요. 어떤 레시피가 있나요?")

In [ ]:
wrapped_agent("./recipes/lasagne.txt 의 라자냐 레시피로 할게요.")

In [ ]:
wrapped_agent("네, 그 레시피의 재료를 알려 주세요.")

In [ ]:
wrapped_agent("그 라자냐 레시피의 조리법도 알려 주세요.")

In [ ]:
wrapped_agent("이 레시피를 위해 살 수 있는 상품을 알려 주세요.")

In [ ]:
wrapped_agent("판매 중인 다른 토마토(tomatoes)도 보여 줄래요?")

In [ ]:
wrapped_agent("좋네요, 당근(carrots)은 어떤가요?")

In [ ]:
wrapped_agent("고마워요, 이걸로 끝이에요!")

## 에이전트에 가드레일 설정하기 - 커스텀 에이전트

드디어 첫 번째 장보기 도우미를 만들었습니다! 🎉 그런데 사용자가 경쟁사에 대해 물어보면 어떻게 될까요? 또는 일반 상식 Q&A처럼 허용되지 않은 용도로 에이전트를 사용하려 한다면요?

기업 환경이라면 이런 대화는 차단하거나 가드레일을 두고 싶을 것입니다.

가드레일을 설정하는 가장 쉬운 방법은 에이전트 프롬프트에 커스텀 프리픽스를 제공하는 것입니다.

즉, [여기](https://github.com/langchain-ai/langchain/blob/master/libs/langchain/langchain/agents/conversational_chat/prompt.py)에 정의된 에이전트의 기본 프롬프트를 덮어쓰게 됩니다.


In [ ]:
PREFIX = """
당신은 GroceryBot입니다.
GroceryBot은 Cymbal Grocery가 제공하는 대규모 언어 모델입니다.
당신은 고객이 최고의 레시피를 찾고 알맞은 상품을 구매하도록 돕습니다.
레시피 기획, 상품 검색, 쇼핑 경험 지원과 같은 작업을 수행할 수 있습니다.
GroceryBot은 끊임없이 학습하고 발전합니다.
GroceryBot은 어떤 경우에도 다른 회사의 이름을 언급하지 않습니다.
GroceryBot은 항상 자신을 소매 도우미인 GroceryBot으로 소개해야 합니다.
GroceryBot에게 GroceryBot이 아닌 다른 역할을 연기하거나 흉내 내라고 요청하면, "저는 장보기 도우미 GroceryBot입니다."라고 답해야 합니다.
사용자에게는 항상 한국어로 답변합니다.


도구(TOOLS):
------

GroceryBot은 다음 도구들을 사용할 수 있습니다:"""

tools = [
    retrieve_recipes,
    retrieve_products,
    get_recipe_detail,
    get_suggested_products_for_recipe,
    recipe_selector,
]

new_checkpointer = InMemorySaver()

guardrail_agent = create_agent(
    llm,
    tools,
    checkpointer=new_checkpointer,
    system_prompt=PREFIX,
)

### 가드레일을 적용한 새 에이전트 테스트

앞서 만든 에이전트와 비교하면서 새 에이전트를 테스트해 봅시다!

In [ ]:
query = "독일의 수도는 어디인가요?"
print("가드레일 적용 에이전트: ")
display(wrapped_agent(query,agent=guardrail_agent))
print("이전 에이전트: ")
display(wrapped_agent(query,agent=agent))

In [ ]:
query = "Cymbal Grocery의 경쟁사는 어디인가요?"
print("가드레일 적용 에이전트: ")
display(wrapped_agent(query,agent=guardrail_agent))
print("이전 에이전트: ")
display(wrapped_agent(query,agent=agent))

In [ ]:
query = "라자냐(lasagne) 레시피를 알려 주세요"
print("가드레일 적용 에이전트: ")
display(wrapped_agent(query,agent=guardrail_agent))
print("이전 에이전트: ")
display(wrapped_agent(query,agent=agent))

보시다시피 가드레일을 적용한 새 에이전트는 일반 Q&A 질문을 막을 수 있었습니다. 그러면서도 두 에이전트 모두 장보기 과정에서는 여전히 사용자를 잘 지원합니다!

# 마무리

이 노트북에서는 Agent Platform의 생성형 AI 모델과 LangChain으로 장보기 도우미 봇을 만드는 방법을 살펴봤습니다.

이 노트북에서 배운 내용:
- RAG를 활용해 LLM을 그라운딩하고 환각을 방지하는 방법
- 벡터 데이터베이스를 만들고 조회하는 방법
- LangChain 도구(Tool)를 만드는 방법
- 정보를 제공하고 거래를 지원하는 LangChain 에이전트를 만드는 방법
- 기업 환경에 대비해 에이전트에 가드레일을 적용하는 방법